# Huấn Luyện Mạng U-Net Chuẩn (Ronneberger 2015) Trên Kaggle
Notebook điều khiển pipeline trên Kaggle GPU (Tesla P100 / T4): Đọc dữ liệu trực tiếp từ Kaggle Dataset `ct208e` -> Huấn luyện -> Đánh giá -> Suy luận -> Tải kết quả.

In [ ]:
# 1. Kiểm tra cấu hình GPU trên Kaggle (P100 16GB hoặc 2x T4)
!nvidia-smi

In [ ]:
# 2. Tự động định vị thư mục dữ liệu từ Kaggle Dataset (/kaggle/input/ct208e/)
import os
from pathlib import Path

# Tự động tìm kiếm vị trí thư mục segmented_slices trên Kaggle
candidates = [
    Path('/kaggle/input/ct208e/segmented_slices'),
    Path('/kaggle/input/ct208e'),
    Path('/content/segmented_slices'),
    Path('../data/segmented_slices')
]

DATA_DIR = None
for c in candidates:
    if (c / 'train' / 'images').exists():
        DATA_DIR = c
        break

if DATA_DIR is None:
    raise FileNotFoundError("❌ Chưa tìm thấy dữ liệu. Hãy đảm bảo bạn đã bấm '+ Add Input' -> chọn Dataset 'ct208e' của bạn trên thanh công cụ bên phải!")

print(f"✅ Đã tìm thấy dữ liệu CT208E tại: {DATA_DIR}")
os.environ['DATA_DIR'] = str(DATA_DIR)


In [ ]:
# 3. Thiết lập thư mục làm việc và mã nguồn tại /kaggle/working
import os, sys, shutil
from pathlib import Path

working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
os.chdir(working_dir)

# Nếu train.py chưa có trong working dir, tự động tìm trong /kaggle/input và copy sang
if not (working_dir / 'train.py').exists() and Path('/kaggle/input').exists():
    for p in Path('/kaggle/input').rglob('train.py'):
        code_src = p.parent
        print(f"📦 Tự động sao chép mã nguồn từ {code_src} sang {working_dir}...")
        for item in code_src.iterdir():
            dst = working_dir / item.name
            if item.is_file() and not dst.exists():
                shutil.copy2(item, dst)
            elif item.is_dir() and not dst.exists() and item.name != 'data':
                shutil.copytree(item, dst)
        break

print(f"📂 Thư mục làm việc hiện tại: {os.getcwd()}")
!ls -lh


In [ ]:
# 4. Cài đặt các thư viện cần thiết
!pip install -r requirements.txt -q


In [ ]:
# 5. Kiểm tra số lượng ảnh & mask trong Dataset trên Kaggle
print('🎉 DỮ LIỆU ĐÃ SẴN SÀNG TRÊN KAGGLE SSD:')
for split in ['train', 'val', 'test']:
    n_i = len(os.listdir(DATA_DIR / split / 'images')) if (DATA_DIR / split / 'images').exists() else 0
    n_m = len(os.listdir(DATA_DIR / split / 'masks')) if (DATA_DIR / split / 'masks').exists() else 0
    print(f'   [{split.upper():5s}] Ảnh: {n_i:4d} | Mask: {n_m:4d}')


In [ ]:
# 6. Bắt đầu quá trình huấn luyện U-Net Chuẩn (Ronneberger 2015)
# =====================================================================
# CÁCH 1 (Khuyên dùng): Huấn luyện TOÀN BỘ 9,343 mẫu (Full Dataset, 50 Epochs, batch_size=16):
!python train.py --config configs/config.yaml \
                 --data.root_dir="$DATA_DIR" \
                 --train.epochs=50 --train.batch_size=16 --train.num_workers=4

# CÁCH 2 (Thử nghiệm nhanh): Huấn luyện với số lượng mẫu giới hạn (5000 mẫu train, 35 epochs):
# !python train.py --config configs/config.yaml \
#                  --data.root_dir="$DATA_DIR" \
#                  --train.limit_train_samples=5000 \
#                  --train.limit_val_samples=500 \
#                  --train.limit_test_samples=500 \
#                  --train.epochs=35 --train.batch_size=16 --train.num_workers=4


In [ ]:
# 7. RESUME huấn luyện (nếu session bị ngắt)
# !python train.py --config configs/config.yaml --data.root_dir="$DATA_DIR" --checkpoint.resume_path=checkpoints/last.pth


In [ ]:
# 8. Trực quan hóa kết quả: Thời gian chạy, Loss và Metric qua từng Epoch
import os
import json
import pandas as pd
import matplotlib.pyplot as plt

history_path = 'checkpoints/history.csv'
summary_path = 'checkpoints/train_summary.json'

if os.path.exists(summary_path):
    with open(summary_path, 'r') as f:
        summary = json.load(f)
    print(f"⏱️ TỔNG THỜI GIAN TRAIN: {summary.get('total_training_time_formatted')}")
    print(f"⚡ Trung bình mỗi Epoch : {summary.get('avg_epoch_time_sec'):.2f} giây")
    print(f"🌟 Epoch tốt nhất       : {summary.get('best_epoch')} (val_dice = {summary.get('best_val_dice'):.4f})")

if os.path.exists(history_path):
    df = pd.read_csv(history_path)
    epochs = df['epoch'].tolist()
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Biểu đồ Loss
    axes[0].plot(epochs, df['train_loss'], 'b-o', label='Train Loss')
    axes[0].plot(epochs, df['val_loss'], 'r-s', label='Val Loss')
    axes[0].set_title('Loss qua từng Epoch', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, linestyle='--', alpha=0.5)
    axes[0].legend()
    
    # 2. Biểu đồ Dice & IoU
    axes[1].plot(epochs, df['val_dice'], 'g-^', label='Val Dice')
    axes[1].plot(epochs, df['val_iou'], 'm-d', label='Val IoU')
    axes[1].set_title('Độ chính xác (Dice & IoU)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Điểm số')
    axes[1].grid(True, linestyle='--', alpha=0.5)
    axes[1].legend()
    
    # 3. Biểu đồ thời gian mỗi Epoch
    if 'time_sec' in df.columns:
        axes[2].bar(epochs, df['time_sec'], color='coral', alpha=0.7)
        axes[2].set_title('Thời gian huấn luyện từng Epoch (s)', fontsize=12, fontweight='bold')
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('Giây (s)')
        axes[2].grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ Chưa tìm thấy history.csv. Hãy hoàn tất huấn luyện ít nhất 1 epoch.')


In [ ]:
# 9. Đánh giá chất lượng mô hình trên tập Test (Toàn bộ 2,006 mẫu (Dice, IoU, HD95))
# ==================================================================================
!python evaluate.py --checkpoint checkpoints/best.pth \
                    --config configs/config.yaml \
                    --data.root_dir="$DATA_DIR" \
                    --threshold=0.5 \
                    --save_overlay


In [ ]:
# 10. Suy luận dự đoán trên tập Test (Inference)
# =====================================================================
!python predict.py --input "$DATA_DIR/test/images" \
                   --output predictions \
                   --checkpoint checkpoints/best.pth \
                   --limit 50


In [ ]:
# 11. Trực quan hóa chuyên sâu (Lọc mẫu có U rõ -> 4 Cột: CT | Bác sĩ | AI | Dán đè)
# =========================================================================================
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

TEST_IMG_DIR = DATA_DIR / 'test' / 'images'
TEST_MSK_DIR = DATA_DIR / 'test' / 'masks'
PRED_MSK_DIR = Path('predictions/masks')

# 1. Lọc các mẫu CÓ NỐT U RÕ RÀNG (diện tích u >= 30 pixels)
candidate_samples = []
pred_files = list(PRED_MSK_DIR.glob('*_mask.png')) if PRED_MSK_DIR.exists() else []

for pf in pred_files:
    stem = pf.stem.replace('_mask', '')
    gt_file = TEST_MSK_DIR / f'{stem}.png'
    if gt_file.exists():
        gt = cv2.imread(str(gt_file), cv2.IMREAD_GRAYSCALE)
        tumor_area = cv2.countNonZero(gt)
        if tumor_area >= 30:
            candidate_samples.append((stem, tumor_area))

candidate_samples.sort(key=lambda x: x[1], reverse=True)
selected_stems = [stem for stem, _ in candidate_samples[:4]]

if not selected_stems and pred_files:
    selected_stems = [pf.stem.replace('_mask', '') for pf in pred_files[:4]]

if not selected_stems:
    print('⚠️ Chưa có ảnh dự đoán nào trong predictions/masks/. Hãy chạy Cell 10 trước.')
else:
    n_samples = len(selected_stems)
    fig, axes = plt.subplots(n_samples, 4, figsize=(18, 4.5 * n_samples))
    if n_samples == 1:
        axes = [axes]

    for row, stem in enumerate(selected_stems):
        img_p = TEST_IMG_DIR / f'{stem}.png'
        gt_p  = TEST_MSK_DIR / f'{stem}.png'
        pr_p  = PRED_MSK_DIR / f'{stem}_mask.png'

        img = cv2.imread(str(img_p), cv2.IMREAD_COLOR)
        gt  = cv2.imread(str(gt_p), cv2.IMREAD_GRAYSCALE) if gt_p.exists() else np.zeros((256, 256), dtype=np.uint8)
        pr  = cv2.imread(str(pr_p), cv2.IMREAD_GRAYSCALE) if pr_p.exists() else np.zeros((256, 256), dtype=np.uint8)

        gt_mask = gt > 0
        pr_mask = pr > 0
        intersection = np.logical_and(gt_mask, pr_mask).sum()
        union = np.logical_or(gt_mask, pr_mask).sum()
        dice = (2.0 * intersection) / (gt_mask.sum() + pr_mask.sum() + 1e-6)
        iou = intersection / (union + 1e-6)

        overlay = img.copy() if img is not None else np.zeros((256, 256, 3), dtype=np.uint8)
        overlap_mask = np.logical_and(gt_mask, pr_mask)
        overlay[gt_mask] = [0, 255, 0]        # Xanh lá: Đáp án bác sĩ
        overlay[pr_mask] = [0, 0, 255]        # Đỏ: AI phân đoạn
        overlay[overlap_mask] = [0, 255, 255] # Vàng: Trùng khớp 100%

        blended = cv2.addWeighted(img, 0.65, overlay, 0.35, 0) if img is not None else overlay

        if img is not None:
            axes[row][0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[row][0].set_title(f'1. CT gốc: {stem}', fontsize=11, fontweight='bold')
        axes[row][0].axis('off')

        axes[row][1].imshow(gt, cmap='gray')
        axes[row][1].set_title(f'2. Bác sĩ vẽ (U: {gt_mask.sum()} px)', fontsize=11, color='green', fontweight='bold')
        axes[row][1].axis('off')

        axes[row][2].imshow(pr, cmap='gray')
        axes[row][2].set_title(f'3. {model_display} (Dice: {dice:.2f} | IoU: {iou:.2f})', fontsize=11, color='blue', fontweight='bold')
        axes[row][2].axis('off')

        axes[row][3].imshow(cv2.cvtColor(blended, cv2.COLOR_BGR2RGB))
        axes[row][3].set_title('4. Dán đè (🟩 Bác sĩ | 🟥 AI | 🟨 Khớp)', fontsize=10, color='darkred', fontweight='bold')
        axes[row][3].axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
# 12. Đóng gói toàn bộ kết quả thành file zip để tải về từ Kaggle Output Pane
# =========================================================================
!zip -q -r /kaggle/working/unet_baseline_results.zip checkpoints/ predictions/ evaluation_results/
print('🎉 ĐÃ ĐÓNG GÓI XONG! Bạn có thể tải file "unet_baseline_results.zip" ngay tại cột Output bên phải.')
